# R Master v6 · True Body Shell（一键断点版）

v5 已锁定：
- 默认衣物是独立 Mesh：Mona_TankTop / Mona_Shorts / Mona_Shoes / Mona_Gloves / Lei_TankTop；
- Mona_Main 自身有 MaskTop / MaskBottom，用来遮住衣服下面的身体；
- v6 因此不再猜材质或连通块。

这版每次渲染前会在**临时场景**里：
1. 只保留 `R2_Mona_Main`；
2. 删除其他所有 Mesh（仅当前 Colab 临时副本，Drive 里的 v2 不会被改）；
3. 关闭 MaskTop / MaskBottom、SurfaceDeform、ParticleSystem；
4. 输出真正的 Body Shell 诊断图。

每张图完成后立刻写入 Google Drive。断线后重新“运行全部”会自动跳过已完成图片。


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, zipfile, json, os

print("R Master v6 · True Body Shell · 断点版")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v2"/"latest"/"R_Master_Align_v2_PREVIEW.blend"
CACHE=ROOT/"cache"
OUT=ROOT/"v6_true_shell"/"latest"
CACHE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)

if not SRC.exists() or SRC.stat().st_size < 50*1024*1024:
    raise RuntimeError("没找到 v2 预览文件。截图给二蛋即可。")

print(f"✓ v2 源：{SRC.stat().st_size/1024/1024:.1f} MiB")



In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v6")
LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size > 100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender Drive 缓存")
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)
    print("✓ Blender 已补充缓存")

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender 就绪")



In [ ]:
SCRIPT=LOCAL/"R_Master_v6_Render.py"
SCRIPT.write_text("\nimport bpy, os, sys, json\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None; view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--view\" and i+1<len(argv): view=argv[i+1]\nif not out or not view: raise RuntimeError(\"missing --out/--view\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nif not body or body.type!=\"MESH\":\n    raise RuntimeError(\"R2_Mona_Main missing\")\n\n# HARD isolation: temporary scene keeps only the duplicated Mona body mesh.\nremoved=[]\nfor obj in list(bpy.data.objects):\n    if obj.type==\"MESH\" and obj != body:\n        removed.append(obj.name)\n        bpy.data.objects.remove(obj, do_unlink=True)\n\n# disable anything that can hide/overlay/externally deform the body shell\ndisabled=[]\nfor mod in body.modifiers:\n    if (\n        mod.type in {\"MASK\",\"SURFACE_DEFORM\",\"CLOTH\",\"PARTICLE_SYSTEM\"}\n        or mod.name in {\"Mask\",\"Mask.001\"}\n        or \"mask\" in mod.name.lower()\n    ):\n        disabled.append({\"name\":mod.name,\"type\":mod.type})\n        mod.show_viewport=False\n        mod.show_render=False\n    elif mod.type in {\"MULTIRES\",\"SUBSURF\"}:\n        mod.levels=min(mod.levels,1)\n        mod.render_levels=min(mod.render_levels,1)\n\nbody.hide_set(False)\nbody.hide_viewport=False\nbody.hide_render=False\n\nfor obj in bpy.context.scene.objects:\n    if obj.type==\"ARMATURE\":\n        obj.hide_set(True)\n        obj.hide_viewport=True\n        obj.hide_render=True\n\nbpy.context.view_layer.update()\n\npts=[body.matrix_world @ Vector(c) for c in body.bound_box]\nmn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\nmx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\ncenter=(mn+mx)*0.5\nheight=mx.z-mn.z; width=mx.x-mn.x; depth=mx.y-mn.y\ndist=max(height,width,depth)*2.5\n\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_WORKBENCH\"\nscene.render.image_settings.file_format=\"PNG\"\nscene.render.film_transparent=False\nscene.display.shading.light=\"STUDIO\"\nscene.display.shading.show_shadows=True\nscene.display.shading.show_cavity=True\nscene.display.shading.cavity_type=\"WORLD\"\nscene.display.shading.color_type=\"SINGLE\"\nscene.display.shading.single_color=(0.60,0.60,0.63)\nscene.display.shading.background_type=\"VIEWPORT\"\nscene.display.shading.background_color=(0.04,0.04,0.05)\n\ncam_data=bpy.data.cameras.get(\"R_Master_v6_Camera_DATA\") or bpy.data.cameras.new(\"R_Master_v6_Camera_DATA\")\ncam=bpy.data.objects.get(\"R_Master_v6_Camera\")\nif not cam:\n    cam=bpy.data.objects.new(\"R_Master_v6_Camera\",cam_data)\n    scene.collection.objects.link(cam)\nscene.camera=cam\ncam.data.type=\"ORTHO\"\n\ndef look_at(obj,target):\n    obj.rotation_euler=(Vector(target)-obj.location).to_track_quat(\"-Z\",\"Y\").to_euler()\n\ndef render_file(fname,pos,target,scale,res):\n    scene.render.resolution_x,scene.render.resolution_y=res\n    scene.render.resolution_percentage=100\n    cam.location=Vector(pos)\n    cam.data.ortho_scale=scale\n    look_at(cam,Vector(target))\n    path=os.path.join(out,fname)\n    scene.render.filepath=path\n    bpy.ops.render.render(write_still=True)\n    return path\n\nspec={}\nspec[\"full_front\"]=(\"R_Master_v6_shell_front.png\",(center.x,center.y-dist,center.z),center,height*1.08,(720,960))\nspec[\"full_side\"]=(\"R_Master_v6_shell_side.png\",(center.x+dist,center.y,center.z),center,height*1.08,(720,960))\nspec[\"full_three_quarter\"]=(\"R_Master_v6_shell_three_quarter.png\",(center.x+dist*.72,center.y-dist*.72,center.z),center,height*1.08,(720,960))\n\nfor key,frac in [(\"torso_waist\",.61),(\"waist_pelvis\",.51),(\"pelvis_upperthigh\",.42)]:\n    z=mn.z+height*frac\n    spec[key]=(f\"R_Master_v6_{key}.png\",(center.x,center.y-dist,z),(center.x,center.y,z),max(.42,height*.25),(900,700))\n\nz=mn.z+height*.46\nspec[\"glute_side\"]=(\"R_Master_v6_glute_side.png\",(center.x+dist,center.y,z),(center.x,center.y,z),max(.42,height*.25),(900,700))\n\nif view not in spec: raise RuntimeError(\"unknown view \"+str(view))\nfname,pos,target,scale,res=spec[view]\npath=render_file(fname,pos,target,scale,res)\n\nreport_path=os.path.join(out,\"R_Master_v6_report.json\")\ndone=sorted([k for k,(f,*_) in spec.items() if os.path.exists(os.path.join(out,f))])\nwith open(report_path,\"w\",encoding=\"utf-8\") as f:\n    json.dump({\n        \"ok\":True,\n        \"stage\":\"R_Master_v6_TrueBodyShell\",\n        \"source_blend\":bpy.data.filepath,\n        \"body_object\":body.name,\n        \"remaining_mesh_objects\":[o.name for o in bpy.data.objects if o.type==\"MESH\"],\n        \"removed_mesh_count\":len(removed),\n        \"removed_mesh_objects\":removed,\n        \"disabled_modifiers\":disabled,\n        \"completed_views\":done,\n        \"total_views\":len(spec),\n        \"rest_pose_baked\":False,\n        \"final_vrm\":False\n    },f,ensure_ascii=False,indent=2)\n\nprint(\"[R Master v6] RENDER_OK\",view,path)\nprint(\"[R Master v6] remaining meshes:\",[o.name for o in bpy.data.objects if o.type==\"MESH\"])\nprint(\"[R Master v6] disabled:\",disabled)\n",encoding="utf-8")
print("✓ v6 真 Body Shell 脚本就绪")



In [ ]:
views=[
 ("full_front","R_Master_v6_shell_front.png"),
 ("full_side","R_Master_v6_shell_side.png"),
 ("full_three_quarter","R_Master_v6_shell_three_quarter.png"),
 ("torso_waist","R_Master_v6_torso_waist.png"),
 ("waist_pelvis","R_Master_v6_waist_pelvis.png"),
 ("pelvis_upperthigh","R_Master_v6_pelvis_upperthigh.png"),
 ("glute_side","R_Master_v6_glute_side.png"),
]

for idx,(view,fname) in enumerate(views,1):
    dest=OUT/fname
    if dest.exists() and dest.stat().st_size>20_000:
        print(f"✓ [{idx}/7] {view} 已存在，跳过")
        continue
    print(f"▶ [{idx}/7] {view}…")
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(SCRIPT),"--","--out",str(OUT),"--view",view]
    p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    important=[x for x in p.stdout.splitlines() if "R Master v6" in x or "Traceback" in x or "Error" in x]
    if important: print("\n".join(important[-12:]))
    if p.returncode!=0:
        raise RuntimeError(f"{view} 失败，退出码 {p.returncode}。截图给二蛋即可。")
    if not dest.exists():
        raise RuntimeError(f"{view} 未生成。")
    print(f"✓ [{idx}/7] 已写入 Drive")
print("✓ v6 七张图全部完成")



In [ ]:
from IPython.display import display,Image,Markdown

views=[
 ("全身正面","R_Master_v6_shell_front.png"),
 ("全身侧面","R_Master_v6_shell_side.png"),
 ("全身 3/4","R_Master_v6_shell_three_quarter.png"),
 ("胸廓→腰","R_Master_v6_torso_waist.png"),
 ("腰→骨盆","R_Master_v6_waist_pelvis.png"),
 ("骨盆→大腿根","R_Master_v6_pelvis_upperthigh.png"),
 ("臀线侧视","R_Master_v6_glute_side.png"),
]
for title,fname in views:
    p=OUT/fname
    if p.exists():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(p),width=500))

review=OUT/"R_Master_v6_Review.zip"
if review.exists(): review.unlink()
with zipfile.ZipFile(review,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for _,fname in views:
        p=OUT/fname
        if p.exists(): z.write(p,arcname=p.name)
    rp=OUT/"R_Master_v6_report.json"
    if rp.exists(): z.write(rp,arcname=rp.name)

print(f"✓ Review ZIP：{review.stat().st_size/1024/1024:.1f} MiB")
files.download(str(review))

